In [0]:
import os
import time
from datetime import datetime
import gc
from dbruntime.databricks_repl_context import get_context

In [0]:
dbutils.widgets.text("batch_control_key","","BATCH_CONTROL_KEY")
batch_control_key=dbutils.widgets.get("batch_control_key").strip()
assert batch_control_key, "BATCH_CONTROL_KEY is required"

In [0]:
ctx=get_context().__dict__
print(ctx)

In [0]:
_orchestrator_path=dbutils.entry_point.getDbutils().notebook().getContext().notebookPath().getOrElse(None)
_base_dir = str(os.path.dirname(_orchestrator_path))
print(_base_dir)


In [0]:
notebooks = [
    {"name": "Bronze to Silver", "path": f"{_base_dir}/bronze_silver"},
    {"name": "Silver to Gold", "path": f"{_base_dir}/silver_gold"}
    # {"name": "Gold to EventHub", "path": f"{_base_dir}/GOLD_EVENTHUB"},
]

# Track execution
execution_summary = []
failed_stage = None

print(f"Pipeline started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"batch_control_key: {batch_control_key}")
print("=" * 70)

for i, notebook in enumerate(notebooks, 1):
    print(f"Executing stage {i} of {len(notebooks)}: {notebook['name']}")
    stage_name=notebook["name"]
    notebook_path=notebook["path"]
    start_time = time.time()
    try:
        print(f"\n[Stage {i}/{len(notebooks)}] Starting: {stage_name}")
        result = dbutils.notebook.run(notebook_path, timeout_seconds=0, arguments={"batch_control_key": batch_control_key})
        duration = round(time.time() - start_time, 2)
        execution_summary.append({"stage": stage_name, "status": "OK", "duration": duration, "result": result})
        print(f"[Stage {i}/{len(notebooks)}] Completed: {stage_name} | Duration: {duration}s | Result: {result}")
        # Memory cleanup between stages - prevents OOM in continuous mode
        # spark.catalog.clearCache()
        gc.collect()

    except Exception as e:
        duration = round(time.time() - start_time, 2)
        execution_summary.append({"stage": stage_name, "status": "FAIL", "duration": duration, "error": str(e)})
        failed_stage = stage_name
        print(f"[Stage {i}/{len(notebooks)}] FAILED: {stage_name} | Duration: {duration}s | Error: {e}")
        break
print("="*80)

In [0]:


# Print Formatted summary
print("\n" + "=" * 70)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 70)

for notebook in notebooks:
    stage_name = notebook["name"]
    entry = next((s for s in execution_summary if s["stage"] == stage_name), None)  #next return distionary
    print(entry)
    if entry:
        status_icon = "[OK]" if entry["status"] == "OK" else "[FAIL]"
        print(f"  {status_icon} {stage_name} - {entry['duration']}s")
    else:
        print(f"  [SKIPPED] {stage_name}")

print("=" * 70)
print(f"Pipeline finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Re-raise exception so continuous mode ON_FAILURE retry kicks in
if failed_stage:
    raise Exception(f"Pipeline Failed at stage: {failed_stage}")
else:
    print("\nAll stages completed successfully.")